In [ ]:
# SECTION 6: Summary and Comparison with Purview

summary = {
    "extraction_timestamp": datetime.utcnow().isoformat(),
    "data_products_count": len(all_data_products),
    "catalog_assets_count": len(all_catalog_assets),
    "data_quality_records": len(all_quality_scores),
    "domains_count": len(all_domains),
    "assets_with_quality_scores": len([s for s in all_quality_scores if s.get("quality_score", 0) > 0])
}

print("\n" + "="*60)
print("UNIFIED CATALOG EXTRACTION SUMMARY")
print("="*60)
for key, value in summary.items():
    print(f"{key}: {value}")

# Store for next notebook
unified_catalog_data = {
    "data_products": all_data_products,
    "catalog_assets": all_catalog_assets,
    "data_quality": all_quality_scores,
    "domains": all_domains,
    "summary": summary
}

print("\n✓ Unified Catalog extraction complete")
print("  Ready for 03-Transform-Data.ipynb to perform relationship mapping")


In [ ]:
# SECTION 5: Extract Domains

def extract_domains(catalog_client: UnifiedCatalogClient) -> List[Dict]:
    """Extract all Domains from Unified Catalog"""
    
    logger.info("Extracting Domains")
    
    try:
        domains = catalog_client.get_domains()
        
        domains_list = []
        for domain in domains:
            record = {
                "domain_id": domain.get("id"),
                "domain_name": domain.get("name"),
                "description": domain.get("description"),
                "owner": domain.get("owner"),
                "parent_domain_id": domain.get("parentDomainId"),
                "created_date": domain.get("createdDate"),
                "product_count": len(domain.get("dataProducts", [])),
                "extracted_at": datetime.utcnow().isoformat(),
                "raw_json": json.dumps(domain)
            }
            domains_list.append(record)
        
        logger.info(f"✓ Extracted {len(domains_list)} Domains")
        return domains_list
    
    except Exception as e:
        logger.error(f"Error extracting Domains: {e}")
        return []

# Extract Domains
try:
    all_domains = extract_domains(catalog_client)
    
    if all_domains:
        domains_df = spark.createDataFrame(all_domains)
        print(f"\n✓ Domains extracted: {len(all_domains)}")
        domains_df.display()
except Exception as e:
    logger.error(f"Cannot extract Domains: {e}")
    all_domains = []


In [ ]:
# SECTION 4: Extract Assets from Data Products

def extract_catalog_assets(catalog_client: UnifiedCatalogClient) -> List[Dict]:
    """Extract all assets from Data Products"""
    
    logger.info("Extracting Catalog assets")
    
    try:
        assets_list = []
        
        # Get all Data Products first
        data_products = catalog_client.get_data_products()
        
        for dp in data_products:
            dp_id = dp.get("id")
            dp_name = dp.get("name")
            
            # Get assets in this Data Product
            assets = catalog_client.get_data_product_assets(dp_id)
            
            for asset in assets:
                record = {
                    "asset_id": asset.get("id"),
                    "asset_name": asset.get("name"),
                    "asset_type": asset.get("type"),
                    "data_product_id": dp_id,
                    "data_product_name": dp_name,
                    "owner": asset.get("owner"),
                    "created_date": asset.get("createdDate"),
                    "tags": ",".join(asset.get("tags", [])),
                    "certified": asset.get("certified", False),
                    "extracted_at": datetime.utcnow().isoformat(),
                    "raw_json": json.dumps(asset)
                }
                assets_list.append(record)
        
        logger.info(f"✓ Extracted {len(assets_list)} Catalog assets")
        return assets_list
    
    except Exception as e:
        logger.error(f"Error extracting Catalog assets: {e}")
        return []

# Extract Catalog assets
try:
    all_catalog_assets = extract_catalog_assets(catalog_client)
    
    if all_catalog_assets:
        assets_df = spark.createDataFrame(all_catalog_assets)
        print(f"\n✓ Catalog assets extracted: {len(all_catalog_assets)}")
        print(f"Asset types: ")
        assets_df.groupby("asset_type").count().show()
except Exception as e:
    logger.error(f"Cannot extract Catalog assets: {e}")
    all_catalog_assets = []


In [ ]:
# SECTION 3: Extract Data Quality Scores

def extract_data_quality(catalog_client: UnifiedCatalogClient) -> List[Dict]:
    """Extract all Data Quality scores from Unified Catalog"""
    
    logger.info("Extracting Data Quality scores")
    
    try:
        quality_scores = catalog_client.get_data_quality_scores()
        
        quality_list = []
        for score_record in quality_scores:
            record = {
                "asset_id": score_record.get("assetId"),
                "asset_name": score_record.get("assetName"),
                "quality_score": float(score_record.get("score", 0)),
                "quality_status": score_record.get("status"),
                "quality_dimensions": score_record.get("dimensions"),
                "last_evaluated": score_record.get("lastEvaluatedDate"),
                "data_product_id": score_record.get("dataProductId"),
                "extracted_at": datetime.utcnow().isoformat(),
                "raw_json": json.dumps(score_record)
            }
            quality_list.append(record)
        
        logger.info(f"✓ Extracted {len(quality_list)} Data Quality records")
        return quality_list
    
    except Exception as e:
        logger.error(f"Error extracting Data Quality scores: {e}")
        return []

# Extract Data Quality
try:
    all_quality_scores = extract_data_quality(catalog_client)
    
    if all_quality_scores:
        quality_df = spark.createDataFrame(all_quality_scores)
        print(f"\n✓ Data Quality scores extracted: {len(all_quality_scores)}")
        
        # Show statistics
        from pyspark.sql.functions import avg, min as spark_min, max as spark_max
        quality_df.agg(
            avg("quality_score").alias("avg_score"),
            spark_min("quality_score").alias("min_score"),
            spark_max("quality_score").alias("max_score")
        ).show()
except Exception as e:
    logger.error(f"Cannot extract Data Quality: {e}")
    all_quality_scores = []


In [ ]:
# SECTION 2: Extract Data Products

def extract_data_products(catalog_client: UnifiedCatalogClient) -> List[Dict]:
    """Extract all Data Products from Unified Catalog"""
    
    logger.info("Extracting Data Products from Unified Catalog")
    
    try:
        data_products = catalog_client.get_data_products()
        
        products_list = []
        for dp in data_products:
            record = {
                "data_product_id": dp.get("id"),
                "data_product_name": dp.get("name"),
                "description": dp.get("description"),
                "owner": dp.get("owner"),
                "domain_id": dp.get("domainId"),
                "created_by": dp.get("createdBy"),
                "created_date": dp.get("createdDate"),
                "tags": ",".join(dp.get("tags", [])),
                "asset_count": len(dp.get("assets", [])),
                "extracted_at": datetime.utcnow().isoformat(),
                "raw_json": json.dumps(dp)
            }
            products_list.append(record)
        
        logger.info(f"✓ Extracted {len(products_list)} Data Products")
        return products_list
    
    except Exception as e:
        logger.error(f"Error extracting Data Products: {e}")
        return []

# Extract Data Products
try:
    catalog_client = UnifiedCatalogClient(workspace_id or "", credential)
    all_data_products = extract_data_products(catalog_client)
    
    if all_data_products:
        dp_df = spark.createDataFrame(all_data_products)
        print(f"\n✓ Data Products extracted: {len(all_data_products)}")
        dp_df.display()
except Exception as e:
    logger.error(f"Cannot connect to Unified Catalog: {e}")
    all_data_products = []


In [ ]:
# SECTION 1: Initialize and Load Fabric Configuration

import sys
import os
from pathlib import Path
import json
import pandas as pd
import logging
from datetime import datetime
from typing import List, Dict, Any

sys.path.insert(0, str(Path.cwd().parent / "src"))

from unified_catalog_client import UnifiedCatalogClient
from config_manager import get_config_manager
from azure.identity import DefaultAzureCredential

logger = logging.getLogger(__name__)

# Get Fabric workspace ID from config or environment
config_mgr = get_config_manager()
workspace_id = os.getenv("FABRIC_WORKSPACE_ID") or config_mgr.get("fabric.workspace_id")
credential = DefaultAzureCredential()

if not workspace_id:
    logger.warning("Fabric workspace ID not configured - using default 'PurviewAuditWorkspace'")
    workspace_id = None  # Will be resolved at runtime

print(f"✓ Unified Catalog extraction initialized")
print(f"  Target Workspace: {workspace_id or 'Will be auto-resolved'}")


# Unified Catalog (Microsoft Fabric) Extract

Extract all metadata from Microsoft Fabric's Unified Catalog including:
- Data Products and ownership
- Assets within each Data Product
- Data Quality scores and metrics
- Domains and domain hierarchy
- Tags and business metadata

**Output**: Standardized Spark DataFrames ready for relationship mapping

**Prerequisites**:
- Fabric workspace with Premium capacity
- 01-Purview-Extract.ipynb completed to establish Purview baseline
